# 04. 카운트형 노드 피처의 월별 드리프트 — 누적 창 vs 고정 창

서비스(v10)는 노드 피처를 **데이터 시작일부터 전체 누적**으로 계산한다. 그러면 `log_out_count` 같은 카운트형 피처는
시간이 갈수록 단조 증가하고, 값이 "계좌 성격"이 아니라 "달력"을 담을 수 있다 (8월 계좌는 데이터가 8/1부터라 전부 신생 계좌처럼 보임).
이 노트북은 그게 실제로 문제인지를 잰다:

> **월별로 세탁 계좌 vs 정상 계좌의 피처 분포 차이(단변량 AUC)가 일정하게 유지되는가?**
> 뒤로 갈수록 AUC 가 떨어지면 누적 창이 신호를 희석하는 것이고, 고정 창(예: 90일)으로 바꿀 근거가 된다.

02 를 여러 번 돌리는 대신, 02 `node_agg` 와 **같은 SQL 정의**로 카운트형 8개만 월말마다 직접 계산한다 (금액 z·reciprocal·통화 수는 제외).

| 피처 (02 이름) | 원시값 | 정의 |
|---|---|---|
| log_out_count / log_in_count | out_n / in_n | 송금 / 수취 건수 |
| log_out_neighbors / log_in_neighbors | out_u / in_u | 서로 다른 수신 / 송신 노드 수 |
| log_active_days | n_days | 송금 또는 수취가 있었던 날짜 수 |
| log_entity_observed_accounts | observed_entity_accounts | 같은 entity_id 로 창 안에서 활동한 계좌 수 |
| log_entity_out_diversity / log_entity_in_diversity | ent_out_n / ent_in_n | 엔티티가 송금한 / 수취한 서로 다른 상대 엔티티 수 |

**창(variant)**: `cumulative` = 데이터 시작~월말 (서비스와 동일), `w90` = 월말 기준 최근 90일 (최장 패턴 70일보다 길게). `VARIANTS` 로 바꿀 수 있다.

**라벨**: 그 달에 활동한 계좌 중, 그 달 안에 세탁 거래(src 또는 dst)에 관여한 계좌 = 양성. 피처값은 그 달 말 기준.
5절은 노드 단위, 7절은 거래(엣지) 단위(모델이 실제로 보는 형태 — 거래의 송신/수신 노드 피처 vs 거래 라벨).

출력: `parquets/<DATASET>/analysis/node_drift*.json`, `node_drift_*.png`. 그림 라벨은 영어(GPU 에 한글 폰트 없음).

## 0. 설정

In [ ]:
!pip install --break-system-packages --no-cache-dir -q duckdb pyarrow matplotlib

In [ ]:
from pathlib import Path
import json, time
import duckdb, pandas as pd, numpy as np
import matplotlib.pyplot as plt

DATASET = "HI-Large"
VARIANTS = {"cumulative": None, "w90": 90}   # 이름 → 창 길이(일). None 이면 데이터 시작부터 누적 (서비스와 동일)
MONTH_ENDS = None                             # None 이면 데이터에서 월말(+마지막 날) 자동. 예: ["2022-09-30", "2022-11-30", "2023-01-12"]
EDGE_SAMPLE_PER_MONTH = 300_000               # 7절 엣지 수준 AUC 에서 라벨0 거래 표본 수 (라벨1 은 전부)
RUN_EDGE_LEVEL = True
TAG = None
THREADS, MEMORY, WORK_DIR = 4, "6GB", None   # Large 는 8 / "24GB" 이상 권장

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "datasets").is_dir())
BASE = ROOT / "parquets" / DATASET
OUT = BASE / "analysis"; OUT.mkdir(parents=True, exist_ok=True)
assert (BASE / "transactions.parquet").is_file() and (BASE / "kyc.parquet").is_file(), "01 먼저 실행"
q = lambda v: "'" + str(v).replace("'", "''") + "'"
suffix = f"_{TAG}" if TAG else ""
fig_path = lambda name: OUT / f"node_drift_{name}{suffix}.png"

work = (Path(WORK_DIR) if WORK_DIR else OUT) / "work_node_drift.duckdb"
for stale in (work, Path(str(work) + ".wal")): stale.unlink(missing_ok=True)
con = duckdb.connect(str(work))
con.execute(f"SET threads={THREADS}"); con.execute(f"SET memory_limit='{MEMORY}'"); con.execute("SET preserve_insertion_order=false")

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})
C_NORMAL, C_LAUND = "#8da0cb", "#e15759"
# 원시값 → 02 피처 이름, 엣지 수준에서 어느 쪽 노드를 볼지
FEATURES = {"out_n": ("log_out_count", "src"), "in_n": ("log_in_count", "dst"),
            "out_u": ("log_out_neighbors", "src"), "in_u": ("log_in_neighbors", "dst"),
            "n_days": ("log_active_days", "src"), "observed_entity_accounts": ("log_entity_observed_accounts", "src"),
            "ent_out_n": ("log_entity_out_diversity", "src"), "ent_in_n": ("log_entity_in_diversity", "dst")}
result = dict(dataset=DATASET, variants=VARIANTS, features=list(FEATURES))
print(ROOT, "\n", OUT)

## 1. 입력 `tx`, `kyc`, 월말 목록

02 와 같은 필터(중복 대표만). `tx` 에 양쪽 entity_id 를 미리 붙여 둔다 (02 `hist` 뷰의 `source_entity/target_entity`).
월말 목록은 데이터에서 자동: 각 달의 마지막 날, 마지막 달은 데이터의 마지막 날.

In [ ]:
t0 = time.perf_counter()
con.execute(f"CREATE OR REPLACE TABLE kyc AS SELECT node_id, entity_id FROM read_parquet({q(BASE/'kyc.parquet')})")
con.execute(f"""
    CREATE OR REPLACE TABLE tx AS
    SELECT t.edge_id, t.day, t.src, t.dst, t.is_laundering, ks.entity_id AS src_entity, kd.entity_id AS dst_entity
    FROM read_parquet({q(BASE/'transactions.parquet')}) t
    JOIN kyc ks ON ks.node_id = t.src JOIN kyc kd ON kd.node_id = t.dst
    WHERE t.duplicate_group_id IS NULL OR t.edge_id = t.duplicate_group_id""")
n, n_l, d0, d1 = con.execute("SELECT count(*), sum(is_laundering::INT), min(day), max(day) FROM tx").fetchone()
if MONTH_ENDS is None:
    MONTH_ENDS = [str(r[0]) for r in con.execute("SELECT DISTINCT least(last_day(day), (SELECT max(day) FROM tx)) AS e FROM tx ORDER BY e").fetchall()]
result.update(rows=n, laundering=n_l, day_min=str(d0), day_max=str(d1), month_ends=MONTH_ENDS)
print(f"거래 {n:,}  라벨1 {n_l:,}  기간 {d0} ~ {d1}   ({time.perf_counter()-t0:.0f}s)")
print("월말:", MONTH_ENDS)

## 2. 월별 라벨 `month_nodes`

달력 월 `[month_start, month_end]` 안에서 활동한(송금 또는 수취) 계좌와, 그 달 세탁 거래에 관여했는지.
활동 안 한 계좌는 그 달 평가에서 제외한다 (모델도 거래가 있는 계좌만 본다).

In [ ]:
con.execute("CREATE OR REPLACE TABLE month_nodes (month_end DATE, node_id BIGINT, laund BOOLEAN)")
for e in MONTH_ENDS:
    con.execute(f"""
        INSERT INTO month_nodes
        SELECT DATE {q(e)}, node_id, bool_or(is_laundering)
        FROM (SELECT src AS node_id, is_laundering FROM tx WHERE day BETWEEN date_trunc('month', DATE {q(e)}) AND DATE {q(e)}
              UNION ALL SELECT dst, is_laundering FROM tx WHERE day BETWEEN date_trunc('month', DATE {q(e)}) AND DATE {q(e)})
        GROUP BY node_id""")
mn = con.execute("SELECT month_end, count(*) AS active_nodes, sum(laund::INT) AS laundering_nodes, avg(laund::INT) AS rate FROM month_nodes GROUP BY 1 ORDER BY 1").df()
result["month_nodes"] = mn.astype({"month_end": str}).to_dict("records")
mn.style.format({"active_nodes": "{:,}", "laundering_nodes": "{:,}", "rate": "{:.3%}"})

## 3. 월말마다 카운트형 노드 피처 계산 `nf`

`node_counts(end, window_days)` 는 02 `node_agg` 의 해당 CTE 를 그대로 쓴다 (`hist` → `WHERE day <= end [AND day > end − W]`).
`observed_entity_accounts` 는 02 처럼 "창 안에서 활동한(`active_days` 에 있는) 같은 entity 의 계좌 수".
variant × 월말 만큼 전체 스캔이 돈다 — Large 는 한 번에 수 분, 2 variant × 6 월말이면 한 시간 안팎.

In [ ]:
def node_counts(end, window_days):
    lower = f"AND day > DATE {q(end)} - INTERVAL {window_days} DAY" if window_days else ""
    return f"""
    WITH h AS (SELECT * FROM tx WHERE day <= DATE {q(end)} {lower}),
         outgoing AS (SELECT src AS node_id, count(*) AS out_n, count(DISTINCT dst) AS out_u FROM h GROUP BY src),
         incoming AS (SELECT dst AS node_id, count(*) AS in_n, count(DISTINCT src) AS in_u FROM h GROUP BY dst),
         active AS (SELECT src AS node_id, day FROM h UNION SELECT dst, day FROM h),
         active_days AS (SELECT node_id, count(*) AS n_days FROM active GROUP BY node_id),
         ent_pairs AS (SELECT DISTINCT src_entity, dst_entity FROM h WHERE src_entity IS NOT NULL AND dst_entity IS NOT NULL),
         ent_out AS (SELECT src_entity AS entity_id, count(*) AS ent_out_n FROM ent_pairs GROUP BY 1),
         ent_in  AS (SELECT dst_entity AS entity_id, count(*) AS ent_in_n  FROM ent_pairs GROUP BY 1)
    SELECT k.node_id,
           coalesce(o.out_n,0) AS out_n, coalesce(i.in_n,0) AS in_n, coalesce(o.out_u,0) AS out_u, coalesce(i.in_u,0) AS in_u, ad.n_days,
           CASE WHEN k.entity_id IS NULL THEN 0 ELSE count(*) OVER (PARTITION BY k.entity_id) END AS observed_entity_accounts,
           coalesce(eo.ent_out_n,0) AS ent_out_n, coalesce(ei.ent_in_n,0) AS ent_in_n
    FROM active_days ad JOIN kyc k USING (node_id) LEFT JOIN outgoing o USING (node_id) LEFT JOIN incoming i USING (node_id)
    LEFT JOIN ent_out eo ON k.entity_id = eo.entity_id LEFT JOIN ent_in ei ON k.entity_id = ei.entity_id"""

con.execute("DROP TABLE IF EXISTS nf")
con.execute("""CREATE TABLE nf (variant VARCHAR, month_end DATE, node_id BIGINT, out_n BIGINT, in_n BIGINT, out_u BIGINT, in_u BIGINT,
                                n_days BIGINT, observed_entity_accounts BIGINT, ent_out_n BIGINT, ent_in_n BIGINT)""")
timing = {}
for variant, w in VARIANTS.items():
    for e in MONTH_ENDS:
        t0 = time.perf_counter()
        con.execute(f"INSERT INTO nf SELECT {q(variant)}, DATE {q(e)}, * FROM ({node_counts(e, w)})")
        timing[f"{variant}@{e}"] = round(time.perf_counter() - t0, 1)
        print(f"{variant:11s} {e}  {timing[f'{variant}@{e}']:6.0f}s")
result["timing_sec"] = timing
con.execute("SELECT variant, month_end, count(*) AS nodes, avg(out_n) AS mean_out_n, max(out_n) AS max_out_n FROM nf GROUP BY 1, 2 ORDER BY 1, 2").df()

## 4. 지표 계산 함수 — 단변량 AUC, 분위수

AUC 는 Mann–Whitney: 값에 평균 순위를 매겨 `(Σrank(양성) − n₊(n₊+1)/2) / (n₊·n₋)`. 0.5 면 구분 못 함, 1 이면 완벽.
0.5 아래면 "값이 작을수록 세탁" 방향 — 신생 계좌 신호는 보통 이쪽으로 나온다. 방향은 `direction` 열(양성 중앙값 − 음성 중앙값의 부호)로 같이 본다.
분위수는 모델 입력과 같은 `ln(1+x)` 스케일.

In [ ]:
def auc_table(data_sql, group_cols, feat_expr, label_col="y"):
    """data_sql 의 각 group_cols 그룹에서 feat_expr 의 단변량 AUC·분위수. feat_expr 는 원시값(ln 변환은 여기서)."""
    g = ", ".join(group_cols)
    return con.execute(f"""
        WITH d AS (SELECT {g}, {label_col} AS y, ln(1 + {feat_expr}) AS v FROM ({data_sql})),
             r AS (SELECT *, rank() OVER (PARTITION BY {g} ORDER BY v) + (count(*) OVER (PARTITION BY {g}, v) - 1) / 2.0 AS rk FROM d)
        SELECT {g},
               count(*) FILTER (WHERE y) AS n_pos, count(*) FILTER (WHERE NOT y) AS n_neg,
               (sum(rk) FILTER (WHERE y) - count(*) FILTER (WHERE y) * (count(*) FILTER (WHERE y) + 1) / 2.0)
                 / (count(*) FILTER (WHERE y) * count(*) FILTER (WHERE NOT y)) AS auc,
               quantile_cont(v, 0.5) FILTER (WHERE y) AS med_pos, quantile_cont(v, 0.5) FILTER (WHERE NOT y) AS med_neg,
               quantile_cont(v, 0.25) FILTER (WHERE y) AS p25_pos, quantile_cont(v, 0.75) FILTER (WHERE y) AS p75_pos,
               quantile_cont(v, 0.25) FILTER (WHERE NOT y) AS p25_neg, quantile_cont(v, 0.75) FILTER (WHERE NOT y) AS p75_neg
        FROM r GROUP BY {g} ORDER BY {g}""").df()

## 5. 노드 수준 — 월별 AUC·분위수

그 달에 활동한 계좌 전체. `auc_abs = |AUC − 0.5|` 가 "분리력"이고, 월별로 이게 유지되는지가 핵심.

In [ ]:
t0 = time.perf_counter()
frames = []
for raw, (name, _) in FEATURES.items():
    d = auc_table("SELECT f.variant, f.month_end, f.{raw} AS val, m.laund AS y FROM nf f JOIN month_nodes m USING (month_end, node_id)".format(raw=raw),
                  ["variant", "month_end"], "val")
    d.insert(0, "feature", name); frames.append(d)
node_auc = pd.concat(frames, ignore_index=True)
node_auc["auc_abs"] = (node_auc.auc - 0.5).abs(); node_auc["direction"] = np.sign(node_auc.med_pos - node_auc.med_neg).map({1: "+", -1: "−", 0: "0"})
node_auc["month_end"] = node_auc.month_end.astype(str)
result["node_level"] = node_auc.round(5).to_dict("records")
print(f"{time.perf_counter()-t0:.0f}s")
# 피벗: 행=피처, 열=월, variant 별
for variant in VARIANTS:
    pv = node_auc[node_auc.variant == variant].pivot(index="feature", columns="month_end", values="auc").reindex([v[0] for v in FEATURES.values()])
    display(pv.style.format("{:.3f}").background_gradient(cmap="RdBu_r", vmin=0.2, vmax=0.8, axis=None).set_caption(f"node-level AUC by month — {variant}"))

In [ ]:
# 그림 A: 월별 AUC (variant 별 패널, 피처별 선)
fig, axes = plt.subplots(1, len(VARIANTS), figsize=(6.5 * len(VARIANTS), 4.2), sharey=True, squeeze=False)
for ax, variant in zip(axes[0], VARIANTS):
    sub = node_auc[node_auc.variant == variant]
    for name in [v[0] for v in FEATURES.values()]:
        s = sub[sub.feature == name].sort_values("month_end")
        ax.plot(s.month_end, s.auc, marker="o", ms=4, label=name)
    ax.axhline(0.5, color="gray", ls="--", lw=1); ax.set_title(f"Node-level univariate AUC by month — {variant}")
    ax.set_xlabel("context_end (month end)"); ax.tick_params(axis="x", rotation=30)
axes[0][0].set_ylabel("AUC (laundering-involved vs other active accounts)"); axes[0][-1].legend(fontsize=7, frameon=False, loc="best")
fig.tight_layout(); fig.savefig(fig_path("A_auc_by_month"), bbox_inches="tight"); plt.show()

In [ ]:
# 그림 B: 피처별 중앙값(ln 스케일) 추이 — 세탁 vs 정상, 누적(실선) vs 고정창(점선)
names = [v[0] for v in FEATURES.values()]
fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharex=True)
styles = dict(zip(VARIANTS, ["-", "--", ":", "-."]))
for ax, name in zip(axes.ravel(), names):
    for variant in VARIANTS:
        s = node_auc[(node_auc.variant == variant) & (node_auc.feature == name)].sort_values("month_end")
        ax.plot(s.month_end, s.med_pos, styles[variant], color=C_LAUND, marker="o", ms=3, label=f"laundering ({variant})")
        ax.plot(s.month_end, s.med_neg, styles[variant], color=C_NORMAL, marker="o", ms=3, label=f"normal ({variant})")
        if variant == list(VARIANTS)[0]:
            ax.fill_between(s.month_end, s.p25_pos, s.p75_pos, color=C_LAUND, alpha=0.10); ax.fill_between(s.month_end, s.p25_neg, s.p75_neg, color=C_NORMAL, alpha=0.10)
    ax.set_title(name, fontsize=10); ax.tick_params(axis="x", rotation=40, labelsize=7)
axes[0][0].set_ylabel("median ln(1+x)"); axes[1][0].set_ylabel("median ln(1+x)"); axes[0][0].legend(fontsize=6, frameon=False)
fig.suptitle(f"Count-type node features over time (band = IQR of first variant) — {DATASET}", y=1.01)
fig.tight_layout(); fig.savefig(fig_path("B_median_by_month"), bbox_inches="tight"); plt.show()

## 6. 요약 — 첫 달 대비 마지막 달의 AUC 변화, 누적 vs 고정 창

`Δauc_abs` 가 음수면 뒤로 갈수록 분리력이 줄어든 것. 같은 달에서 `w90 − cumulative` 가 양수면 고정 창이 더 잘 가른다.

In [ ]:
first, last = node_auc.month_end.min(), node_auc.month_end.max()
rows = []
for name in names:
    r = {"feature": name}
    for variant in VARIANTS:
        s = node_auc[(node_auc.variant == variant) & (node_auc.feature == name)].set_index("month_end")
        r[f"{variant}_first"] = s.loc[first, "auc_abs"]; r[f"{variant}_last"] = s.loc[last, "auc_abs"]; r[f"{variant}_Δ"] = s.loc[last, "auc_abs"] - s.loc[first, "auc_abs"]
    if len(VARIANTS) >= 2:
        a, b = list(VARIANTS)[:2]
        r[f"{b}−{a} (last)"] = r[f"{b}_last"] - r[f"{a}_last"]
    rows.append(r)
summary = pd.DataFrame(rows).set_index("feature")
result["node_level_summary"] = summary.round(5).to_dict("index")
summary.style.format("{:+.3f}").background_gradient(cmap="RdBu", vmin=-0.1, vmax=0.1, subset=[c for c in summary.columns if "Δ" in c or "−" in c])

## 7. 거래(엣지) 수준 — 모델이 보는 형태 (선택)

그 달의 거래마다 송신 노드(out·entity 계열) 또는 수신 노드(in 계열)의 피처값을 붙이고, **거래 라벨**에 대한 AUC.
라벨1 거래는 전부, 라벨0 은 월별 `EDGE_SAMPLE_PER_MONTH` 건 표본. 노드 수준과 결론이 같은지 확인용.

In [ ]:
if RUN_EDGE_LEVEL:
    t0 = time.perf_counter()
    con.execute("DROP TABLE IF EXISTS edge_smp")
    con.execute("CREATE TABLE edge_smp (month_end DATE, edge_id BIGINT, src BIGINT, dst BIGINT, y BOOLEAN)")
    for e in MONTH_ENDS:
        n_norm = con.execute(f"SELECT count(*) FROM tx WHERE NOT is_laundering AND day BETWEEN date_trunc('month', DATE {q(e)}) AND DATE {q(e)}").fetchone()[0]
        p = min(1.0, EDGE_SAMPLE_PER_MONTH / max(n_norm, 1))
        con.execute(f"""INSERT INTO edge_smp SELECT DATE {q(e)}, edge_id, src, dst, is_laundering FROM tx
                        WHERE day BETWEEN date_trunc('month', DATE {q(e)}) AND DATE {q(e)} AND (is_laundering OR random() < {p})""")
    frames = []
    for raw, (name, side) in FEATURES.items():
        d = auc_table(f"""SELECT f.variant, s.month_end, f.{raw} AS val, s.y
                          FROM edge_smp s JOIN nf f ON f.month_end = s.month_end AND f.node_id = s.{side}""", ["variant", "month_end"], "val")
        d.insert(0, "feature", name); d.insert(1, "node_side", side); frames.append(d)
    edge_auc = pd.concat(frames, ignore_index=True)
    edge_auc["auc_abs"] = (edge_auc.auc - 0.5).abs(); edge_auc["month_end"] = edge_auc.month_end.astype(str)
    result["edge_level"] = edge_auc.round(5).to_dict("records")
    print(f"{time.perf_counter()-t0:.0f}s")
    for variant in VARIANTS:
        pv = edge_auc[edge_auc.variant == variant].pivot(index="feature", columns="month_end", values="auc").reindex(names)
        display(pv.style.format("{:.3f}").background_gradient(cmap="RdBu_r", vmin=0.2, vmax=0.8, axis=None).set_caption(f"edge-level AUC by month — {variant}"))
    fig, axes = plt.subplots(1, len(VARIANTS), figsize=(6.5 * len(VARIANTS), 4.2), sharey=True, squeeze=False)
    for ax, variant in zip(axes[0], VARIANTS):
        sub = edge_auc[edge_auc.variant == variant]
        for name in names:
            s = sub[sub.feature == name].sort_values("month_end"); ax.plot(s.month_end, s.auc, marker="o", ms=4, label=name)
        ax.axhline(0.5, color="gray", ls="--", lw=1); ax.set_title(f"Edge-level univariate AUC by month — {variant}"); ax.tick_params(axis="x", rotation=30)
    axes[0][0].set_ylabel("AUC (laundering tx vs sampled normal tx)"); axes[0][-1].legend(fontsize=7, frameon=False)
    fig.tight_layout(); fig.savefig(fig_path("C_edge_auc_by_month"), bbox_inches="tight"); plt.show()
else:
    edge_auc = None; print("RUN_EDGE_LEVEL=False — 7절 건너뜀")

## 8. 마무리 — `analysis/node_drift<tag>.json`, 작업 파일 정리

In [ ]:
con.close()
for stale in (work, Path(str(work) + ".wal")): stale.unlink(missing_ok=True)
(OUT / f"node_drift{suffix}.json").write_text(json.dumps(result, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
print(summary.round(3).to_string())
for f in sorted(OUT.glob(f"node_drift*{suffix}.*")): print(f"{f.name:44s} {f.stat().st_size/1e3:8.1f} KB")

## 읽는 법 / 다음 결정

- **그림 A / 5절 표**: 피처별 AUC 선이 월별로 평평하면 누적 창이어도 분리력이 유지되는 것. 뒤로 갈수록 0.5 쪽으로 오면 누적이 신호를 희석한다.
- **그림 B**: 정상 계좌 중앙값이 매달 올라가는 건 누적 창의 당연한 모양. 중요한 건 **빨강(세탁)과 파랑(정상) 사이 간격**이 유지되는지. 점선(w90)은 11월 이후 평평해져야 정상(90일이 차면 더 안 쌓임).
- **요약 표**: `cumulative_Δ` 가 −0.03 아래로 떨어지는 피처가 있고 같은 달 `w90 − cumulative` 가 양수면 → 그 피처는 고정 창 전환 근거. 전부 ±0.01 안이면 데이터셋 기간(5.5개월) 안에선 누적 창 그대로 둬도 됨.
- **7절**이 5절과 다른 결론이면 엣지 수준을 우선한다 (모델이 실제로 받는 형태).
- 주의: 8월은 데이터 시작 직후라 누적·w90 이 같고 모든 계좌가 "신생"으로 보인다. 8월 AUC 는 기준점이라기보다 왜곡 구간으로 보고, 9월 이후 추세를 본다.
- 결론이 "고정 창"이면 02 를 `WINDOW_DAYS=90, TAG="w90"` 로 돌려 19개 노드 피처 전체를 만들고, 학습·서비스(`preprocess.py` 의 `prepare_snapshot` 하한) 를 **같이** 바꿔야 한다. 운영이 길어지면(1년+) 누적 창은 학습 분포 밖으로 가므로 데이터셋 안에서 괜찮더라도 서비스용은 고정 창을 권한다.